In [14]:
import json
from pathlib import Path

dataset_name = "Baby"  # choices: Beauty, Baby, Clothing

DATASET_CONFIGS = {
    "Beauty": {
        "data_dir": "Beauty",
        "review_files": ["All_Beauty.jsonl"],
        "meta_files": ["meta_All_Beauty.jsonl"],
    },
    "Baby": {
        "data_dir": "Baby",
        "review_files": ["All_Baby.jsonl", "Baby_Products.jsonl"],
        "meta_files": ["meta_All_Baby.jsonl", "meta_Baby_Products.jsonl"],
    },
    "Clothing": {
        "data_dir": "Clothing",
        "review_files": ["All_Clothing.jsonl", "Clothing_Shoes_and_Jewelry.jsonl"],
        "meta_files": ["meta_All_Clothing.jsonl", "meta_Clothing_Shoes_and_Jewelry.jsonl"],
    },
}

def first_existing_path(data_dir, filenames, label):
    for filename in filenames:
        path = data_dir / filename
        if path.exists():
            return path
    tried = ", ".join(str(data_dir / filename) for filename in filenames)
    raise FileNotFoundError(f"{label} file not found. Tried: {tried}")

if dataset_name not in DATASET_CONFIGS:
    choices = ", ".join(DATASET_CONFIGS)
    raise ValueError(f"Unknown dataset_name: {dataset_name}. Choices: {choices}")

dataset_config = DATASET_CONFIGS[dataset_name]
data_dir = Path(dataset_config["data_dir"])
meta_path = first_existing_path(data_dir, dataset_config["meta_files"], "Meta")

meta_dict = {}

with meta_path.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        parent_asin = record.get("parent_asin")
        if parent_asin is not None:
            meta_dict[parent_asin] = record

print(f"Dataset: {dataset_name}")
print(f"Meta items: {len(meta_dict)}")

Dataset: Baby
Meta items: 217724


In [15]:
import csv
import json
from collections import Counter
from pathlib import Path

if "dataset_name" not in globals():
    dataset_name = "Baby"  # choices: Beauty, Baby, Clothing

DATASET_CONFIGS = {
    "Beauty": {
        "data_dir": "Beauty",
        "review_files": ["All_Beauty.jsonl"],
        "meta_files": ["meta_All_Beauty.jsonl"],
    },
    "Baby": {
        "data_dir": "Baby",
        "review_files": ["All_Baby.jsonl", "Baby_Products.jsonl"],
        "meta_files": ["meta_All_Baby.jsonl", "meta_Baby_Products.jsonl"],
    },
    "Clothing": {
        "data_dir": "Clothing",
        "review_files": ["All_Clothing.jsonl", "Clothing_Shoes_and_Jewelry.jsonl"],
        "meta_files": ["meta_All_Clothing.jsonl", "meta_Clothing_Shoes_and_Jewelry.jsonl"],
    },
}

def first_existing_path(data_dir, filenames, label):
    for filename in filenames:
        path = data_dir / filename
        if path.exists():
            return path
    tried = ", ".join(str(data_dir / filename) for filename in filenames)
    raise FileNotFoundError(f"{label} file not found. Tried: {tried}")

if dataset_name not in DATASET_CONFIGS:
    choices = ", ".join(DATASET_CONFIGS)
    raise ValueError(f"Unknown dataset_name: {dataset_name}. Choices: {choices}")

dataset_config = DATASET_CONFIGS[dataset_name]
data_dir = Path(dataset_config["data_dir"])
review_path = first_existing_path(data_dir, dataset_config["review_files"], "Review")
output_prefix = review_path.stem
output_path = data_dir / f"{output_prefix}_5core_interactions.csv"
user_output_path = data_dir / f"{output_prefix}_5core_users.csv"
item_output_path = data_dir / f"{output_prefix}_5core_items.csv"

min_rating = 0.0
min_interactions = 5

interactions = []
with review_path.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        user_id = record.get("user_id")
        parent_asin = record.get("parent_asin")
        rating = record.get("rating")
        if (
            user_id is not None
            and parent_asin is not None
            and rating is not None
        ):
            interactions.append((user_id, parent_asin, float(rating)))

rating_interactions = [
    (user_id, parent_asin)
    for user_id, parent_asin, rating in interactions
    if rating >= min_rating
]

filtered_interactions = rating_interactions
while True:
    user_counts = Counter(user_id for user_id, _ in filtered_interactions)
    item_counts = Counter(parent_asin for _, parent_asin in filtered_interactions)

    next_interactions = [
        (user_id, parent_asin)
        for user_id, parent_asin in filtered_interactions
        if user_counts[user_id] >= min_interactions
        and item_counts[parent_asin] >= min_interactions
    ]

    if len(next_interactions) == len(filtered_interactions):
        break
    filtered_interactions = next_interactions

raw_users = sorted({user_id for user_id, _, _ in interactions})
raw_items = sorted({parent_asin for _, parent_asin, _ in interactions})
prefilter_users = sorted({user_id for user_id, _ in rating_interactions})
prefilter_items = sorted({parent_asin for _, parent_asin in rating_interactions})
filtered_users = sorted({user_id for user_id, _ in filtered_interactions})
filtered_items = sorted({parent_asin for _, parent_asin in filtered_interactions})

def interaction_density(num_interactions, num_users, num_items):
    if num_users == 0 or num_items == 0:
        return 0.0
    return num_interactions / (num_users * num_items)

with output_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["user_id", "parent_asin"])
    writer.writerows(filtered_interactions)

with user_output_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["user_id"])
    writer.writerows((user_id,) for user_id in filtered_users)

with item_output_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["parent_asin"])
    writer.writerows((parent_asin,) for parent_asin in filtered_items)

print(f"Raw interactions: {len(interactions)}")
print(f"Raw users: {len(raw_users)}")
print(f"Raw items: {len(raw_items)}")
print(f"Raw density: {interaction_density(len(interactions), len(raw_users), len(raw_items)):.8f}")
print(f"After rating filter: {len(rating_interactions)}")
print(f"After rating users: {len(prefilter_users)}")
print(f"After rating items: {len(prefilter_items)}")
print(f"After rating density: {interaction_density(len(rating_interactions), len(prefilter_users), len(prefilter_items)):.8f}")
print(f"Filtered interactions: {len(filtered_interactions)}")
print(f"Filtered users: {len(filtered_users)}")
print(f"Filtered items: {len(filtered_items)}")
print(f"Filtered density: {interaction_density(len(filtered_interactions), len(filtered_users), len(filtered_items)):.8f}")
print(f"Saved interactions to: {output_path}")
print(f"Saved users to: {user_output_path}")
print(f"Saved items to: {item_output_path}")

Raw interactions: 6028884
Raw users: 3386206
Raw items: 217654
Raw density: 0.00000818
After rating filter: 6028884
After rating users: 3386206
After rating items: 217654
After rating density: 0.00000818
Filtered interactions: 1287653
Filtered users: 155434
Filtered items: 36845
Filtered density: 0.00022484
Saved interactions to: Baby/All_Baby_5core_interactions.csv
Saved users to: Baby/All_Baby_5core_users.csv
Saved items to: Baby/All_Baby_5core_items.csv


In [1]:
import csv
import json
from pathlib import Path

if "dataset_name" not in globals():
    dataset_name = "Baby"  # choices: Beauty, Baby, Clothing

DATASET_CONFIGS = {
    "Beauty": {
        "data_dir": "Beauty",
        "review_files": ["All_Beauty.jsonl"],
        "meta_files": ["meta_All_Beauty.jsonl"],
    },
    "Baby": {
        "data_dir": "Baby",
        "review_files": ["All_Baby.jsonl", "Baby_Products.jsonl"],
        "meta_files": ["meta_All_Baby.jsonl", "meta_Baby_Products.jsonl"],
    },
    "Clothing": {
        "data_dir": "Clothing",
        "review_files": ["All_Clothing.jsonl", "Clothing_Shoes_and_Jewelry.jsonl"],
        "meta_files": ["meta_All_Clothing.jsonl", "meta_Clothing_Shoes_and_Jewelry.jsonl"],
    },
}

def first_existing_path(data_dir, filenames, label):
    for filename in filenames:
        path = data_dir / filename
        if path.exists():
            return path
    tried = ", ".join(str(data_dir / filename) for filename in filenames)
    raise FileNotFoundError(f"{label} file not found. Tried: {tried}")

def first_large_image(images):
    if not isinstance(images, list):
        return None
    for image in images:
        if isinstance(image, dict) and image.get("large"):
            return image["large"]
    return None

if dataset_name not in DATASET_CONFIGS:
    choices = ", ".join(DATASET_CONFIGS)
    raise ValueError(f"Unknown dataset_name: {dataset_name}. Choices: {choices}")

dataset_config = DATASET_CONFIGS[dataset_name]
data_dir = Path(dataset_config["data_dir"])
review_path = first_existing_path(data_dir, dataset_config["review_files"], "Review")
meta_path = first_existing_path(data_dir, dataset_config["meta_files"], "Meta")
output_prefix = review_path.stem
output_path = data_dir / f"{output_prefix}_5core_interactions.csv"
user_output_path = data_dir / f"{output_prefix}_5core_users.csv"
item_output_path = data_dir / f"{output_prefix}_5core_items.csv"

interactions = []
with output_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        user_id = row.get("user_id")
        parent_asin = row.get("parent_asin")
        if user_id and parent_asin:
            interactions.append((user_id, parent_asin))

candidate_items = {parent_asin for _, parent_asin in interactions}
item_meta = {}
with meta_path.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        parent_asin = record.get("parent_asin")
        if parent_asin not in candidate_items:
            continue

        title = record.get("title")
        image = first_large_image(record.get("images"))
        if title and image:
            item_meta[parent_asin] = {
                "title": title,
                "image": image,
            }

filtered_interactions = [
    (user_id, parent_asin)
    for user_id, parent_asin in interactions
    if parent_asin in item_meta
]
filtered_users = sorted({user_id for user_id, _ in filtered_interactions})
filtered_items = sorted({parent_asin for _, parent_asin in filtered_interactions})

with output_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["user_id", "parent_asin"])
    writer.writerows(filtered_interactions)

with user_output_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["user_id"])
    writer.writerows((user_id,) for user_id in filtered_users)

with item_output_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["parent_asin", "title", "image"])
    writer.writerows(
        (parent_asin, item_meta[parent_asin]["title"], item_meta[parent_asin]["image"])
        for parent_asin in filtered_items
    )

print(f"Dataset: {dataset_name}")
print(f"Original interactions: {len(interactions)}")
print(f"Original items: {len(candidate_items)}")
print(f"Items with title and large image: {len(item_meta)}")
print(f"Filtered interactions: {len(filtered_interactions)}")
print(f"Filtered users: {len(filtered_users)}")
print(f"Filtered items: {len(filtered_items)}")
print(f"Overwrote interactions: {output_path}")
print(f"Overwrote users: {user_output_path}")
print(f"Overwrote items: {item_output_path}")

Dataset: Baby
Original interactions: 1287653
Original items: 36845
Items with title and large image: 36838
Filtered interactions: 1287278
Filtered users: 155434
Filtered items: 36838
Overwrote interactions: Baby/All_Baby_5core_interactions.csv
Overwrote users: Baby/All_Baby_5core_users.csv
Overwrote items: Baby/All_Baby_5core_items.csv


In [ ]:
import csv
import mimetypes
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

if "dataset_name" not in globals():
    dataset_name = "Baby"  # choices: Beauty, Baby, Clothing

DATASET_CONFIGS = {
    "Beauty": {
        "data_dir": "Beauty",
        "review_files": ["All_Beauty.jsonl"],
    },
    "Baby": {
        "data_dir": "Baby",
        "review_files": ["All_Baby.jsonl", "Baby_Products.jsonl"],
    },
    "Clothing": {
        "data_dir": "Clothing",
        "review_files": ["All_Clothing.jsonl", "Clothing_Shoes_and_Jewelry.jsonl"],
    },
}

def first_existing_path(data_dir, filenames, label):
    for filename in filenames:
        path = data_dir / filename
        if path.exists():
            return path
    tried = ", ".join(str(data_dir / filename) for filename in filenames)
    raise FileNotFoundError(f"{label} file not found. Tried: {tried}")

def image_extension(url, content_type):
    path_suffix = Path(urlparse(url).path).suffix.lower()
    if path_suffix in {".jpg", ".jpeg", ".png", ".webp", ".gif"}:
        return path_suffix

    guessed_suffix = mimetypes.guess_extension(content_type.split(";")[0].strip()) if content_type else None
    if guessed_suffix in {".jpg", ".jpeg", ".png", ".webp", ".gif"}:
        return guessed_suffix

    return ".jpg"

if dataset_name not in DATASET_CONFIGS:
    choices = ", ".join(DATASET_CONFIGS)
    raise ValueError(f"Unknown dataset_name: {dataset_name}. Choices: {choices}")

dataset_config = DATASET_CONFIGS[dataset_name]
data_dir = Path(dataset_config["data_dir"])
review_path = first_existing_path(data_dir, dataset_config["review_files"], "Review")
output_prefix = review_path.stem
item_output_path = data_dir / f"{output_prefix}_5core_items.csv"
image_dir = data_dir / "image"
image_dir.mkdir(parents=True, exist_ok=True)

timeout = 20
downloaded = 0
skipped = 0
failed = []

with item_output_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        parent_asin = row.get("parent_asin")
        image_url = row.get("image")
        if not parent_asin or not image_url:
            failed.append((parent_asin, image_url, "missing parent_asin or image"))
            continue

        existing_files = list(image_dir.glob(f"{parent_asin}.*"))
        if existing_files:
            skipped += 1
            continue

        request = Request(
            image_url,
            headers={"User-Agent": "Mozilla/5.0"},
        )
        try:
            with urlopen(request, timeout=timeout) as response:
                content = response.read()
                content_type = response.headers.get("Content-Type", "")
                suffix = image_extension(image_url, content_type)
                image_path = image_dir / f"{parent_asin}{suffix}"
                image_path.write_bytes(content)
                downloaded += 1
        except (HTTPError, URLError, TimeoutError, OSError) as error:
            failed.append((parent_asin, image_url, repr(error)))

print(f"Dataset: {dataset_name}")
print(f"Image directory: {image_dir}")
print(f"Downloaded: {downloaded}")
print(f"Skipped existing: {skipped}")
print(f"Failed: {len(failed)}")
if failed:
    print("First failed rows:")
    for row in failed[:10]:
        print(row)

In [ ]:
import csv
import mimetypes
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen

if "dataset_name" not in globals():
    dataset_name = "Baby"  # choices: Beauty, Baby, Clothing

DATASET_CONFIGS = {
    "Beauty": {"data_dir": "Beauty", "review_files": ["All_Beauty.jsonl"]},
    "Baby": {"data_dir": "Baby", "review_files": ["All_Baby.jsonl", "Baby_Products.jsonl"]},
    "Clothing": {"data_dir": "Clothing", "review_files": ["All_Clothing.jsonl", "Clothing_Shoes_and_Jewelry.jsonl"]},
}

def first_existing_path(data_dir, filenames, label):
    for filename in filenames:
        path = data_dir / filename
        if path.exists():
            return path
    tried = ", ".join(str(data_dir / filename) for filename in filenames)
    raise FileNotFoundError(f"{label} file not found. Tried: {tried}")

def image_exists(image_dir, parent_asin):
    return any(path.is_file() and path.stat().st_size > 0 for path in image_dir.glob(f"{parent_asin}.*"))

def image_extension(url, content_type):
    suffix = Path(urlparse(url).path).suffix.lower()
    if suffix in {".jpg", ".jpeg", ".png", ".webp", ".gif"}:
        return suffix
    guessed = mimetypes.guess_extension(content_type.split(";")[0].strip()) if content_type else None
    return guessed if guessed in {".jpg", ".jpeg", ".png", ".webp", ".gif"} else ".jpg"

def write_log(path, rows):
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["parent_asin", "title", "image", "error"])
        writer.writeheader()
        writer.writerows(rows)

if dataset_name not in DATASET_CONFIGS:
    choices = ", ".join(DATASET_CONFIGS)
    raise ValueError(f"Unknown dataset_name: {dataset_name}. Choices: {choices}")

data_dir = Path(DATASET_CONFIGS[dataset_name]["data_dir"])
review_path = first_existing_path(data_dir, DATASET_CONFIGS[dataset_name]["review_files"], "Review")
output_prefix = review_path.stem
items_path = data_dir / f"{output_prefix}_5core_items.csv"
image_dir = data_dir / "image"
missing_log_path = data_dir / f"{output_prefix}_missing_images.csv"
failed_log_path = data_dir / f"{output_prefix}_failed_image_downloads.csv"
image_dir.mkdir(parents=True, exist_ok=True)

with items_path.open("r", encoding="utf-8", newline="") as f:
    items = list(csv.DictReader(f))

missing_items = []
for item in items:
    parent_asin = item.get("parent_asin")
    if not parent_asin or not image_exists(image_dir, parent_asin):
        missing_items.append({
            "parent_asin": parent_asin,
            "title": item.get("title", ""),
            "image": item.get("image", ""),
            "error": "missing before download",
        })

write_log(missing_log_path, missing_items)
print(f"Total items: {len(items)}", flush=True)
print(f"Existing images: {len(items) - len(missing_items)}", flush=True)
print(f"Missing images before download: {len(missing_items)}", flush=True)
print(f"Missing log: {missing_log_path}", flush=True)

timeout = 10
downloaded = 0
skipped = 0
failed_items = []

for idx, item in enumerate(missing_items, start=1):
    parent_asin = item.get("parent_asin")
    image_url = item.get("image")

    if parent_asin and image_exists(image_dir, parent_asin):
        skipped += 1
        print(f"{idx}/{len(missing_items)} skipped existing {parent_asin}", flush=True)
        continue

    if not parent_asin or not image_url:
        item["error"] = "missing parent_asin or image URL"
        failed_items.append(item)
        print(f"{idx}/{len(missing_items)} failed missing fields {parent_asin}", flush=True)
        continue

    try:
        request = Request(image_url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(request, timeout=timeout) as response:
            content = response.read()
            content_type = response.headers.get("Content-Type", "")
        if not content:
            raise ValueError("empty image response")
        suffix = image_extension(image_url, content_type)
        image_path = image_dir / f"{parent_asin}{suffix}"
        image_path.write_bytes(content)
        downloaded += 1
        print(f"{idx}/{len(missing_items)} downloaded {parent_asin} bytes={len(content)}", flush=True)
    except (HTTPError, URLError, TimeoutError, OSError, ValueError) as error:
        item["error"] = repr(error)
        failed_items.append(item)
        print(f"{idx}/{len(missing_items)} failed {parent_asin}: {error!r}", flush=True)

write_log(failed_log_path, failed_items)
print(f"Downloaded this run: {downloaded}", flush=True)
print(f"Skipped this run: {skipped}", flush=True)
print(f"Failed this run: {len(failed_items)}", flush=True)
print(f"Failed log: {failed_log_path}", flush=True)
